## 1. Dataset Audit and Initial Inspection

The first step is to locate and inspect the dataset available in the Kaggle environment.

This stage verifies the dataset directory structure, identifies the available classification categories, and calculates the number of images in each category.

### 1.1 Automatic Dataset Detection

Instead of manually specifying the dataset path, the notebook searches the Kaggle input directory (`/kaggle/input`) for matching `train` and `test` folders.

The dataset is accepted only when exactly one matching dataset directory is found. This helps prevent accidental selection of an incorrect dataset.

### 1.2 Dataset Structure

The expected directory structure is:

```text
dataset/
├── train/
│   ├── class_1/
│   ├── class_2/
│   └── ...
│
└── test/
    ├── class_1/
    ├── class_2/
    └── ...
```

Each class directory contains images belonging to a specific skin disease category.

### 1.3 Image File Extensions

The audit recognizes the following image formats:

- JPG
- JPEG
- PNG
- WEBP

Only files directly inside each class directory are counted.

### 1.4 Class Distribution Analysis

The notebook calculates:

- Number of training categories
- Number of testing categories
- Total training images
- Total testing images
- Image count for each category
- Whether training and testing class names match

### 1.5 Dataset Validation

Matching class names between the training and testing directories is important because the model must use a consistent class-to-index mapping during training and evaluation.

**Expected Dataset Summary:**

| Dataset Split | Images |
|---|---:|
| Training | 15,557 |
| Testing | 4,002 |
| Total Categories | 23 |

The dataset audit establishes the initial structure and class distribution before image quality inspection and preprocessing.

In [1]:
from pathlib import Path
from collections import Counter

# ==========================================
# VITAMIN DEFICIENCY AI - DATASET AUDIT
# ==========================================

INPUT_DIR = Path("/kaggle/input")

# Automatically locate train and test directories
matches = []

for train_dir in INPUT_DIR.rglob("train"):
    test_dir = train_dir.parent / "test"

    if train_dir.is_dir() and test_dir.is_dir():
        matches.append(train_dir.parent)

if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one dataset, found {len(matches)}: {matches}"
    )

DATA_DIR = matches[0]
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def get_image_count(folder):
    return sum(
        1 for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

print("=" * 60)
print("VITAMIN DEFICIENCY AI - DATASET AUDIT")
print("=" * 60)

print(f"\nDataset path: {DATA_DIR}")

train_classes = sorted(
    d.name for d in TRAIN_DIR.iterdir() if d.is_dir()
)

test_classes = sorted(
    d.name for d in TEST_DIR.iterdir() if d.is_dir()
)

train_counts = {
    name: get_image_count(TRAIN_DIR / name)
    for name in train_classes
}

test_counts = {
    name: get_image_count(TEST_DIR / name)
    for name in test_classes
}

print(f"\nTraining classes: {len(train_classes)}")
print(f"Testing classes:  {len(test_classes)}")

print(f"\nTraining images: {sum(train_counts.values()):,}")
print(f"Testing images:  {sum(test_counts.values()):,}")

print("\nCLASS DISTRIBUTION")
print("-" * 60)

for name in train_classes:
    print(
        f"{name}: "
        f"Train={train_counts[name]:,} | "
        f"Test={test_counts.get(name, 0):,}"
    )

print("\nCLASS MATCH:", set(train_classes) == set(test_classes))
print("\nDATASET AUDIT COMPLETED")

VITAMIN DEFICIENCY AI - DATASET AUDIT

Dataset path: /kaggle/input/datasets/satyearth/vitamin-deficiency

Training classes: 23
Testing classes:  23

Training images: 15,557
Testing images:  4,002

CLASS DISTRIBUTION
------------------------------------------------------------
Acne and Rosacea Photos: Train=840 | Test=312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions: Train=1,149 | Test=288
Atopic Dermatitis Photos: Train=489 | Test=123
Bullous Disease Photos: Train=448 | Test=113
Cellulitis Impetigo and other Bacterial Infections: Train=288 | Test=73
Eczema Photos: Train=1,235 | Test=309
Exanthems and Drug Eruptions: Train=404 | Test=101
Hair Loss Photos Alopecia and other Hair Diseases: Train=239 | Test=60
Herpes HPV and other STDs Photos: Train=405 | Test=102
Light Diseases and Disorders of Pigmentation: Train=568 | Test=143
Lupus and other Connective Tissue diseases: Train=420 | Test=105
Melanoma Skin Cancer Nevi and Moles: Train=463 | Test=116
Nail Fungus and o

## 2. Image Sample Quality Inspection

After verifying the dataset structure, a sample-based image quality inspection is performed to identify potentially corrupted or unreadable image files.

### 2.1 Sampling Strategy

To avoid scanning every image in the dataset, five images are randomly selected from each training category.

A fixed random seed (`42`) is used to make the sampling process reproducible.

With 23 categories and five samples per category, up to 115 images are inspected.

### 2.2 Image Verification

The Python Imaging Library (PIL) is used to verify the selected images.

Each sampled image is checked for:

- File readability and integrity
- Image mode (RGB, RGBA, grayscale, etc.)
- Image format (JPEG, PNG, WEBP, etc.)
- Image dimensions (width and height)

The `Image.verify()` method checks whether an image file is structurally valid.

### 2.3 Corrupted Image Detection

If an image cannot be opened or verified, its file path is recorded as a potentially corrupted sample.

This helps identify possible dataset quality issues before preprocessing and model training.

### 2.4 Inspection Scope

This is a sample-based quality check, not an exhaustive scan of the complete dataset.

Only selected training images are inspected. Therefore, zero corrupted samples means that no corruption was detected among the inspected samples; it does not guarantee that every dataset image is valid.

**Outcome:** The inspection provides an initial assessment of image integrity, formats, color modes, and dimensions before building the data-loading pipeline.

In [2]:
import random
from PIL import Image
from collections import Counter

# ==========================================
# OPTIMIZED IMAGE SAMPLE CHECK
# ==========================================

random.seed(42)

sample_size = 5
checked = 0
corrupted = []
image_modes = Counter()
image_formats = Counter()
image_sizes = Counter()

for class_name in train_classes:
    class_files = [
        f for f in (TRAIN_DIR / class_name).iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ]

    samples = random.sample(
        class_files,
        min(sample_size, len(class_files))
    )

    for file in samples:
        try:
            with Image.open(file) as img:
                img.verify()

            with Image.open(file) as img:
                checked += 1
                image_modes[img.mode] += 1
                image_formats[img.format] += 1
                image_sizes[img.size] += 1

        except Exception:
            corrupted.append(str(file))

print("=" * 55)
print("IMAGE SAMPLE QUALITY REPORT")
print("=" * 55)

print(f"Images checked: {checked}")
print(f"Corrupted samples: {len(corrupted)}")
print(f"Image modes: {dict(image_modes)}")
print(f"Image formats: {dict(image_formats)}")

print("\nImage dimensions:")
for size, count in image_sizes.most_common(10):
    print(f"{size}: {count}")

if corrupted:
    print("\nCorrupted examples:")
    for file in corrupted:
        print(file)

print("\nSAMPLE CHECK COMPLETED")

IMAGE SAMPLE QUALITY REPORT
Images checked: 115
Corrupted samples: 0
Image modes: {'RGB': 115}
Image formats: {'JPEG': 115}

Image dimensions:
(720, 472): 27
(720, 480): 17
(480, 720): 9
(720, 464): 3
(467, 720): 2
(720, 475): 2
(473, 720): 2
(720, 488): 2
(720, 477): 2
(472, 720): 2

SAMPLE CHECK COMPLETED


## 3. Environment Setup and Reproducibility

This stage prepares the deep learning environment and configures the libraries, computational device, and training parameters required for the project.

### 3.1 Libraries and Frameworks

The following libraries are imported:

- **PyTorch:** Model development and training.
- **Torchvision:** Pretrained architectures and image transformations.
- **NumPy:** Numerical operations and array processing.
- **Scikit-learn:** Dataset splitting and classification evaluation.
- **PIL:** Image loading and inspection.
- **Python Standard Libraries:** Randomization, operating system operations, and utility functions.

### 3.2 Reproducibility

A fixed random seed of `42` is configured for Python, NumPy, and PyTorch.

This helps make random operations more consistent across executions, including dataset sampling and model initialization.

Note that complete reproducibility is not guaranteed across different hardware and software environments.

### 3.3 Project Configuration

| Parameter | Value |
|---|---:|
| Input Image Size | 224 × 224 |
| Batch Size | 32 |
| DataLoader Workers | 2 |
| Initial Learning Rate | 0.0001 |
| Initial Training Epochs | 30 |
| Number of Classes | 23 |
| Random Seed | 42 |

### 3.4 Hardware Configuration

The notebook automatically detects whether a CUDA-compatible GPU is available.

- CUDA GPU is selected when available.
- CPU is used as a fallback.
- GPU name and available memory are displayed when CUDA is enabled.

### 3.5 Environment Verification

The environment report displays the installed versions of PyTorch, Torchvision, and NumPy, along with the selected computational device and project configuration.

**Outcome:** The computational environment is configured and ready for dataset preprocessing, model construction, and training.

In [3]:
import os
import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# ==========================================
# REPRODUCIBILITY
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ==========================================
# PROJECT CONFIGURATION
# ==========================================

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
LEARNING_RATE = 0.0001
EPOCHS = 30

NUM_CLASSES = len(train_classes)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ==========================================
# ENVIRONMENT REPORT
# ==========================================

print("=" * 55)
print("VITAMIN DEFICIENCY AI - ENVIRONMENT")
print("=" * 55)

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Device: {DEVICE}")
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(f"\nImage Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Workers: {NUM_WORKERS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Number of Classes: {NUM_CLASSES}")

print("\nENVIRONMENT CONFIGURED SUCCESSFULLY")

VITAMIN DEFICIENCY AI - ENVIRONMENT
PyTorch Version: 2.11.0+cu130
Torchvision Version: 0.26.0+cu130
NumPy Version: 2.1.3
Device: cuda
GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB

Image Size: 224x224
Batch Size: 32
Workers: 2
Learning Rate: 0.0001
Epochs: 30
Number of Classes: 23

ENVIRONMENT CONFIGURED SUCCESSFULLY


## 4. Data Preparation and Preprocessing

In this stage, the image dataset is loaded, transformed, divided into training and validation subsets, and prepared using PyTorch DataLoaders.

### 4.1 Training Image Transformations

Data augmentation is applied to training images to introduce visual variations and help the model generalize better.

The training transformations include:

| Transformation | Configuration | Purpose |
|---|---|---|
| Random Resized Crop | 224 × 224, scale 0.8–1.0 | Introduces variations in image framing |
| Random Horizontal Flip | Probability 0.5 | Adds horizontal variations |
| Random Rotation | ±10 degrees | Introduces slight orientation changes |
| Color Jitter | Brightness 0.1, Contrast 0.1 | Adds mild lighting and contrast variations |
| ToTensor | — | Converts images into PyTorch tensors |
| Normalize | ImageNet mean and standard deviation | Normalizes input channels |

ImageNet normalization is used because the project uses an ImageNet-pretrained EfficientNet-B0 model.

### 4.2 Validation and Testing Transformations

Validation and testing images use deterministic preprocessing without random augmentation.

The transformations include:

1. Resize the image to 256 pixels.
2. Apply a center crop of 224 × 224.
3. Convert the image into a tensor.
4. Apply ImageNet normalization.

Using consistent evaluation transformations ensures that validation and test images are processed in the same way.

### 4.3 Dataset Loading

PyTorch's `ImageFolder` is used to load images based on their class directory names.

Three dataset objects are created:

- `train_base`: Training images with augmentation.
- `val_base`: The same training images with evaluation preprocessing.
- `test_dataset`: Separate test images with evaluation preprocessing.

Class mapping consistency is verified using assertions.

### 4.4 Stratified Train-Validation Split

The original training dataset is divided into:

| Split | Percentage | Images |
|---|---:|---:|
| Training | 85% | 13,223 |
| Validation | 15% | 2,334 |

Stratified sampling preserves the approximate class distribution in both subsets.

The split uses a fixed random seed of `42`.

The validation subset uses the evaluation transformation rather than training augmentation.

### 4.5 DataLoaders

PyTorch DataLoaders efficiently provide batches of images during training and evaluation.

| Configuration | Training | Validation |
|---|---|---|
| Batch Size | 32 | 32 |
| Shuffle | Enabled | Disabled |
| Workers | 2 | 2 |
| Pin Memory | Enabled | Enabled |
| Persistent Workers | Enabled | Enabled |

A seeded generator is used for training DataLoader shuffling.

### 4.6 Dataset Summary

The prepared datasets contain:

- 13,223 training images
- 2,334 validation images
- 4,002 independent test images
- 23 matching classification categories

**Outcome:** The datasets are ready for model training, validation, and final evaluation.

In [4]:
from torch.utils.data import Subset

# ==========================================
# DATA TRANSFORMS
# ==========================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ==========================================
# LOAD DATASETS
# ==========================================

train_base = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=train_transform
)

val_base = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    root=str(TEST_DIR),
    transform=eval_transform
)

assert train_base.classes == val_base.classes
assert train_base.classes == test_dataset.classes

# ==========================================
# STRATIFIED TRAIN / VALIDATION SPLIT
# ==========================================

targets = np.array(train_base.targets)
indices = np.arange(len(targets))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(train_base, train_indices)
val_dataset = Subset(val_base, val_indices)

# ==========================================
# DATALOADERS
# ==========================================

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

# ==========================================
# DATASET REPORT
# ==========================================

print("=" * 55)
print("DATA PREPARATION REPORT")
print("=" * 55)

print(f"Total Training Images: {len(train_base):,}")
print(f"Training Images: {len(train_dataset):,}")
print(f"Validation Images: {len(val_dataset):,}")
print(f"Testing Images: {len(test_dataset):,}")

print(f"\nNumber of Classes: {len(train_base.classes)}")
print(f"Training Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")

print("\nClass Mapping:")
for index, name in enumerate(train_base.classes):
    print(f"{index}: {name}")

print("\nDATA PREPARATION COMPLETED")

DATA PREPARATION REPORT
Total Training Images: 15,557
Training Images: 13,223
Validation Images: 2,334
Testing Images: 4,002

Number of Classes: 23
Training Batches: 414
Validation Batches: 73

Class Mapping:
0: Acne and Rosacea Photos
1: Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions
2: Atopic Dermatitis Photos
3: Bullous Disease Photos
4: Cellulitis Impetigo and other Bacterial Infections
5: Eczema Photos
6: Exanthems and Drug Eruptions
7: Hair Loss Photos Alopecia and other Hair Diseases
8: Herpes HPV and other STDs Photos
9: Light Diseases and Disorders of Pigmentation
10: Lupus and other Connective Tissue diseases
11: Melanoma Skin Cancer Nevi and Moles
12: Nail Fungus and other Nail Disease
13: Poison Ivy Photos and other Contact Dermatitis
14: Psoriasis pictures Lichen Planus and related diseases
15: Scabies Lyme Disease and other Infestations and Bites
16: Seborrheic Keratoses and other Benign Tumors
17: Systemic Disease
18: Tinea Ringworm Candidiasis and ot

## 5. EfficientNet-B0 Model Architecture

This project uses **EfficientNet-B0**, a convolutional neural network architecture, as the backbone for skin disease image classification.

The model is initialized with pretrained ImageNet weights using Torchvision.

### 5.1 Transfer Learning

Transfer learning allows a model trained on a large dataset to reuse its previously learned visual representations for a new classification task.

The pretrained EfficientNet-B0 backbone provides useful feature extraction capabilities, reducing the need to train the entire network from scratch.

### 5.2 Backbone Freezing

During the initial training stage, all parameters inside the feature extraction layers (`model.features`) are frozen.

This means:

- Pretrained feature extraction weights remain unchanged.
- Only the newly introduced classification head is trained.
- The initial training focuses on adapting the classifier to the 23 target categories.

### 5.3 Custom Classification Head

The original ImageNet classification head is replaced with a custom classifier:

```python
nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES)
)
```

The classifier contains:

| Layer | Configuration | Purpose |
|---|---|---|
| Dropout | 0.3 | Reduces overfitting |
| Linear | 1,280 → 23 | Produces class scores |

The output layer generates 23 logits, one for each skin disease category.

### 5.4 Model Parameters

The model contains approximately:

| Parameter Type | Count |
|---|---:|
| Total Parameters | 4,037,011 |
| Trainable Parameters | 29,463 |
| Frozen Parameters | 4,007,548 |

Only the classification head is trainable during this initial stage.

### 5.5 Device Configuration

The initialized model is moved to the previously selected computational device, allowing GPU-accelerated training when CUDA is available.

**Outcome:** An ImageNet-pretrained EfficientNet-B0 model is customized for 23-class skin disease image classification, with its backbone frozen for initial training.

In [5]:
# ==========================================
# EFFICIENTNET-B0 MODEL
# ==========================================

from torchvision.models import EfficientNet_B0_Weights

# Load pretrained ImageNet weights
weights = EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

# Freeze backbone for initial training
for param in model.features.parameters():
    param.requires_grad = False

# Replace classification head
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES)
)

# Move model to GPU
model = model.to(DEVICE)

# ==========================================
# MODEL SUMMARY
# ==========================================

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("=" * 55)
print("EFFICIENTNET-B0 MODEL REPORT")
print("=" * 55)

print(f"Architecture: EfficientNet-B0")
print(f"Pretrained: ImageNet")
print(f"Output Classes: {NUM_CLASSES}")
print(f"Device: {DEVICE}")

print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {total_params - trainable_params:,}")

print("\nClassification Head:")
print(model.classifier)

print("\nMODEL INITIALIZED SUCCESSFULLY")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 128MB/s] 


EFFICIENTNET-B0 MODEL REPORT
Architecture: EfficientNet-B0
Pretrained: ImageNet
Output Classes: 23
Device: cuda

Total Parameters: 4,037,011
Trainable Parameters: 29,463
Frozen Parameters: 4,007,548

Classification Head:
Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=23, bias=True)
)

MODEL INITIALIZED SUCCESSFULLY


## 6. Loss Function, Class Weights and Optimizer

This stage configures the loss function, optimizer, learning rates, learning rate scheduler, and mixed precision training.

### 6.1 Class Frequency Calculation

The number of training samples in each category is calculated using only the training subset.

This information is used to assign class weights to the loss function.

### 6.2 Class Weight Calculation

The project uses square-root inverse frequency weighting:

\[
w_i = \sqrt{\frac{N}{C \times n_i}}
\]

Where:

- \(w_i\) = Weight assigned to class \(i\)
- \(N\) = Total number of training samples
- \(C\) = Total number of classes
- \(n_i\) = Number of training samples in class \(i\)

Classes with fewer training samples receive larger weights, while more frequent classes receive smaller weights.

Square-root weighting provides a moderate adjustment rather than using full inverse-frequency weighting.

### 6.3 Weighted Cross-Entropy Loss

`nn.CrossEntropyLoss` is configured with the calculated class weights.

This encourages the model to pay more attention to classification errors involving underrepresented categories.

### 6.4 AdamW Optimizer

The AdamW optimizer is used to update the trainable classification head during the initial training stage.

| Parameter | Value |
|---|---:|
| Optimizer | AdamW |
| Head Learning Rate | 0.001 |
| Fine-Tuning Learning Rate | 0.00001 |
| Weight Decay | 0.0001 |

The fine-tuning learning rate is configured for later training stages.

### 6.5 Learning Rate Scheduler

`ReduceLROnPlateau` monitors validation loss.

Its configuration:

- Mode: Minimum validation loss
- Reduction factor: 0.5
- Patience: 3 epochs

When validation loss stops improving for the configured patience period, the learning rate is reduced.

### 6.6 Automatic Mixed Precision

Automatic Mixed Precision (AMP) is enabled when CUDA is available.

The GradScaler helps maintain numerical stability during mixed-precision gradient updates.

This can reduce GPU memory usage and improve training efficiency.

### 6.7 Configuration Summary

The notebook prints the training class counts and corresponding weights, along with the optimizer, loss function, learning rates, scheduler, and AMP configuration.

**Outcome:** The initial head-training configuration is ready, with weighted loss to address class imbalance and AdamW optimization for model updates.

In [6]:
# ==========================================
# LOSS FUNCTION & OPTIMIZER
# ==========================================

# Calculate training class frequencies
train_targets = np.array(train_base.targets)[train_indices]

class_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES
)

# Balanced class weights
class_weights = np.sqrt(
    len(train_targets) / (NUM_CLASSES * class_counts)
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

# Weighted Cross Entropy
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Initial training: classification head only
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
WEIGHT_DECAY = 1e-4

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# Mixed precision support
USE_AMP = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

# ==========================================
# CONFIGURATION REPORT
# ==========================================

print("=" * 55)
print("TRAINING CONFIGURATION")
print("=" * 55)

print(f"Loss Function: Weighted CrossEntropyLoss")
print(f"Optimizer: AdamW")
print(f"Head Learning Rate: {HEAD_LR}")
print(f"Fine-tuning Learning Rate: {FINE_TUNE_LR}")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"Scheduler: ReduceLROnPlateau")
print(f"Mixed Precision: {USE_AMP}")

print("\nCLASS WEIGHTS")
print("-" * 55)

for index, name in enumerate(train_base.classes):
    print(
        f"{index:02d} | "
        f"Count: {class_counts[index]:5d} | "
        f"Weight: {class_weights[index].item():.4f} | "
        f"{name}"
    )

print("\nTRAINING CONFIGURATION READY")

TRAINING CONFIGURATION
Loss Function: Weighted CrossEntropyLoss
Optimizer: AdamW
Head Learning Rate: 0.001
Fine-tuning Learning Rate: 1e-05
Weight Decay: 0.0001
Scheduler: ReduceLROnPlateau
Mixed Precision: True

CLASS WEIGHTS
-------------------------------------------------------
00 | Count:   714 | Weight: 0.8973 | Acne and Rosacea Photos
01 | Count:   977 | Weight: 0.7671 | Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions
02 | Count:   416 | Weight: 1.1756 | Atopic Dermatitis Photos
03 | Count:   381 | Weight: 1.2284 | Bullous Disease Photos
04 | Count:   245 | Weight: 1.5319 | Cellulitis Impetigo and other Bacterial Infections
05 | Count:  1050 | Weight: 0.7400 | Eczema Photos
06 | Count:   343 | Weight: 1.2947 | Exanthems and Drug Eruptions
07 | Count:   203 | Weight: 1.6829 | Hair Loss Photos Alopecia and other Hair Diseases
08 | Count:   344 | Weight: 1.2928 | Herpes HPV and other STDs Photos
09 | Count:   483 | Weight: 1.0910 | Light Diseases and Disorders of

## 7. Training and Validation Engine

This stage implements reusable training and validation functions for the EfficientNet-B0 classification model.

The same training engine is reused across multiple training stages to maintain consistency and avoid duplicating training logic.

### 7.1 Training Function

The `train_one_epoch()` function performs one complete training epoch.

Its main operations include:

1. Set the model to training mode.
2. Load image batches and move them to the selected device.
3. Reset the optimizer gradients.
4. Perform forward propagation.
5. Calculate the classification loss.
6. Perform backward propagation.
7. Apply gradient clipping.
8. Update model parameters using the optimizer.
9. Calculate average loss and training accuracy.

### 7.2 Validation Function

The `validate_one_epoch()` function evaluates model performance on the validation dataset without updating model parameters.

It uses:

- Evaluation mode through `model.eval()`.
- `torch.inference_mode()` to disable gradient tracking.
- The same configured loss function.
- Validation loss and accuracy calculations.

Validation is performed without optimizer updates.

### 7.3 Automatic Mixed Precision

Both functions use PyTorch Automatic Mixed Precision when CUDA is available.

The training function uses `GradScaler` to help maintain numerical stability during gradient updates.

The validation function uses autocast without gradient computation.

### 7.4 Gradient Clipping

Gradient clipping is applied with a maximum norm of `1.0`.

This helps control excessively large gradients and can improve training stability.

### 7.5 Progress Monitoring

`tqdm` provides progress bars during training and validation.

The current batch loss is displayed to monitor the optimization process.

### 7.6 Returned Metrics

Both functions return:

- Average loss across the processed dataset.
- Accuracy across the processed dataset.

**Outcome:** A reusable training and validation engine is initialized for all subsequent model training stages.

In [7]:
# ==========================================
# TRAINING & VALIDATION ENGINE
# ==========================================

from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Training", leave=False)

    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(1) == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / total, correct / total


@torch.inference_mode()
def validate_one_epoch(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Validation", leave=False)

    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(1) == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / total, correct / total


print("=" * 55)
print("TRAINING ENGINE INITIALIZED")
print("=" * 55)

print("Training function: Ready")
print("Validation function: Ready")
print("Mixed precision: Enabled")
print("Gradient clipping: Enabled")
print("\nTRAINING ENGINE READY")

TRAINING ENGINE INITIALIZED
Training function: Ready
Validation function: Ready
Mixed precision: Enabled
Gradient clipping: Enabled

TRAINING ENGINE READY


## 8. Training Loop Configuration and Checkpoint Management

Before starting model training, the notebook initializes the training history, checkpoint path, and early stopping configuration.

### 8.1 Training History

A dictionary named `history` stores the metrics collected during training.

The recorded metrics include:

| Metric | Description |
|---|---|
| Train Loss | Training loss for each epoch |
| Validation Loss | Validation loss for each epoch |
| Train Accuracy | Training accuracy for each epoch |
| Validation Accuracy | Validation accuracy for each epoch |
| Learning Rate | Optimizer learning rate for each epoch |

These records allow training progress to be reviewed and compared later.

### 8.2 Best Model Checkpoint

The best model checkpoint is configured to be saved at:

`/kaggle/working/best_skin_disease_model.pth`

The checkpoint strategy is based on the lowest validation loss.

This helps preserve the model state that performs best on the validation dataset rather than automatically keeping the final training epoch.

### 8.3 Early Stopping

Early stopping is configured with a patience of 7 epochs.

If validation loss does not improve for seven consecutive epochs, the training loop can stop early.

This helps avoid unnecessary training when validation performance stops improving.

### 8.4 Learning Rate Scheduling

The previously configured `ReduceLROnPlateau` scheduler is used to adjust the learning rate based on validation loss.

### 8.5 Initial State

The best validation loss is initialized to infinity, and the best epoch is initialized to zero.

This ensures that the first valid improvement can be saved as the initial best checkpoint.

**Outcome:** The training process is configured with metric tracking, early stopping, learning rate scheduling, and best-checkpoint management.

In [8]:
# ==========================================
# TRAINING LOOP
# ==========================================

import copy
import time

BEST_MODEL_PATH = "/kaggle/working/best_skin_disease_model.pth"

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "learning_rate": []
}

best_val_loss = float("inf")
best_epoch = 0
patience = 7
epochs_without_improvement = 0

print("=" * 55)
print("TRAINING LOOP CONFIGURATION")
print("=" * 55)

print(f"Total Epochs: {EPOCHS}")
print(f"Early Stopping Patience: {patience}")
print(f"Best Model Path: {BEST_MODEL_PATH}")
print("Checkpoint Strategy: Best Validation Loss")
print("Scheduler: ReduceLROnPlateau")
print("\nTRAINING LOOP READY")

TRAINING LOOP CONFIGURATION
Total Epochs: 30
Early Stopping Patience: 7
Best Model Path: /kaggle/working/best_skin_disease_model.pth
Checkpoint Strategy: Best Validation Loss
Scheduler: ReduceLROnPlateau

TRAINING LOOP READY


## 9. Initial Model Training

This stage performs the initial training of the EfficientNet-B0 model using the training and validation datasets.

Since the feature extraction backbone was frozen earlier, only the newly added classification head is updated during this training stage.

### 9.1 Epoch-Based Training

The model is trained for a maximum of `EPOCHS` epochs.

During each epoch:

1. The model is trained using `train_one_epoch()`.
2. Training loss and accuracy are calculated.
3. Validation is performed using `validate_one_epoch()`.
4. The learning rate scheduler is updated using validation loss.
5. Training metrics and the current learning rate are recorded in the `history` dictionary.
6. Epoch execution time is measured and displayed.

### 9.2 Best Model Checkpoint

After each epoch, the validation loss is compared with the best validation loss recorded so far.

- If validation loss improves, the model checkpoint is saved.
- The best epoch and validation loss are updated.
- The early stopping counter is reset.
- If validation loss does not improve, the counter increases.

The checkpoint contains:

- Best epoch number
- Model state dictionary
- Optimizer state dictionary
- Best validation loss
- Class names
- Input image size

### 9.3 Early Stopping

Training stops automatically if validation loss fails to improve for seven consecutive epochs.

This prevents unnecessary training after validation performance stops improving.

### 9.4 Training Summary

After training finishes, the notebook reports:

- Best epoch
- Best validation loss
- Total training time
- Saved checkpoint location

**Outcome:** The initial classifier head is trained, and the checkpoint with the lowest validation loss is preserved for subsequent model development.

In [9]:
# ==========================================
# START TRAINING
# ==========================================

training_start_time = time.time()

for epoch in range(EPOCHS):

    epoch_start_time = time.time()

    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_accuracy = validate_one_epoch(
        model, val_loader, criterion
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)
    history["learning_rate"].append(current_lr)

    epoch_time = time.time() - epoch_start_time

    print(f"Train Loss:     {train_loss:.4f}")
    print(f"Validation Loss:{val_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Val Accuracy:   {val_accuracy * 100:.2f}%")
    print(f"Learning Rate:  {current_lr:.7f}")
    print(f"Epoch Time:     {epoch_time / 60:.2f} minutes")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "class_names": train_classes,
            "image_size": IMAGE_SIZE
        }, BEST_MODEL_PATH)

        print("Best model checkpoint saved!")

    else:
        epochs_without_improvement += 1
        print(
            f"No improvement: "
            f"{epochs_without_improvement}/{patience}"
        )

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break

total_training_time = (time.time() - training_start_time) / 60

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Total Training Time: {total_training_time:.2f} minutes")
print(f"Checkpoint: {BEST_MODEL_PATH}")


EPOCH 1/30


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss:     2.6382
Validation Loss:2.4839
Train Accuracy: 26.01%
Val Accuracy:   29.61%
Learning Rate:  0.0010000
Epoch Time:     11.24 minutes
Best model checkpoint saved!

EPOCH 2/30


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

... (81 outputs left out)

Train Loss:     2.0587
Validation Loss:2.2864
Train Accuracy: 39.30%
Val Accuracy:   35.18%
Learning Rate:  0.0005000
Epoch Time:     1.81 minutes
No improvement: 2/7

EPOCH 30/30


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss:     2.0631
Validation Loss:2.2826
Train Accuracy: 38.92%
Val Accuracy:   34.83%
Learning Rate:  0.0005000
Epoch Time:     1.82 minutes
No improvement: 3/7

TRAINING COMPLETED
Best Epoch: 27
Best Validation Loss: 2.2778
Total Training Time: 64.03 minutes
Checkpoint: /kaggle/working/best_skin_disease_model.pth


## 10. Fine-Tuning EfficientNet-B0

After the initial head-only training, the best checkpoint is restored and the model enters the fine-tuning stage.

The objective is to adapt the deeper pretrained EfficientNet-B0 feature representations to the skin disease image classification task.

### 10.1 Restore the Best Checkpoint

The checkpoint with the lowest validation loss from the initial training stage is loaded.

This ensures that fine-tuning starts from the best-performing initial model rather than simply continuing from the final epoch.

### 10.2 Selective Layer Unfreezing

The feature extractor is initially frozen again.

Then, the last two feature blocks are unfrozen:

- Earlier feature blocks remain frozen.
- The last two feature blocks become trainable.
- The classification head remains trainable.

This approach preserves general pretrained image features while allowing deeper layers to adapt to the target dataset.

### 10.3 Differential Learning Rates

AdamW is configured with separate learning rates for the feature extractor and classifier.

| Model Component | Learning Rate |
|---|---:|
| Last two EfficientNet-B0 feature blocks | 0.00001 |
| Classification head | 0.0001 |

A lower learning rate is used for pretrained feature layers to reduce the risk of disrupting previously learned representations.

Weight decay is retained from the earlier optimizer configuration.

### 10.4 Fine-Tuning Configuration

| Setting | Value |
|---|---:|
| Maximum fine-tuning epochs | 10 |
| Early stopping patience | 4 |
| Scheduler | ReduceLROnPlateau |
| Scheduler factor | 0.5 |
| Scheduler patience | 2 |
| Optimizer | AdamW |

### 10.5 Validation and Checkpointing

Each fine-tuning epoch performs training followed by validation.

The learning rate scheduler monitors validation loss.

Whenever validation loss improves, the model state is saved to:

`/kaggle/working/best_finetuned_skin_model.pth`

Fine-tuning stops early if validation loss does not improve for four consecutive epochs.

### 10.6 Expected Outcome

This stage produces a fine-tuned EfficientNet-B0 checkpoint containing the best validation-performing model from the fine-tuning process.

**Outcome:** The model is further adapted to the 23-class skin disease dataset using selective backbone unfreezing and differential learning rates.

In [10]:
# ==========================================
# FINE-TUNE EFFICIENTNET-B0
# ==========================================

# Restore the best checkpoint
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

# Freeze all backbone layers first
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the last two feature blocks
for param in model.features[-2:].parameters():
    param.requires_grad = True

# Train classifier and unfrozen feature blocks
optimizer = torch.optim.AdamW(
    [
        {"params": model.features[-2:].parameters(),
         "lr": FINE_TUNE_LR},
        {"params": model.classifier.parameters(),
         "lr": 1e-4}
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

FINE_TUNE_EPOCHS = 10
FINE_TUNE_PATIENCE = 4

fine_best_loss = float("inf")
fine_best_epoch = 0
fine_wait = 0

FINE_TUNED_PATH = "/kaggle/working/best_finetuned_skin_model.pth"

print("Restored checkpoint from epoch:", checkpoint["epoch"])
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Fine-tuning configuration ready.")

for epoch in range(FINE_TUNE_EPOCHS):

    start = time.time()

    print(f"\nFINE-TUNING EPOCH {epoch + 1}/{FINE_TUNE_EPOCHS}")

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_accuracy = validate_one_epoch(
        model, val_loader, criterion
    )

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
    print(f"Epoch Time: {(time.time() - start) / 60:.2f} minutes")

    if val_loss < fine_best_loss:
        fine_best_loss = val_loss
        fine_best_epoch = epoch + 1
        fine_wait = 0

        torch.save({
            "epoch": fine_best_epoch,
            "model_state_dict": model.state_dict(),
            "best_val_loss": fine_best_loss,
            "class_names": train_classes,
            "image_size": IMAGE_SIZE
        }, FINE_TUNED_PATH)

        print("Fine-tuned checkpoint saved!")

    else:
        fine_wait += 1
        print(f"No improvement: {fine_wait}/{FINE_TUNE_PATIENCE}")

    if fine_wait >= FINE_TUNE_PATIENCE:
        print("Fine-tuning early stopping triggered.")
        break

print("\nFINE-TUNING COMPLETED")
print("Best Fine-Tuning Epoch:", fine_best_epoch)
print("Best Fine-Tuning Validation Loss:", round(fine_best_loss, 4))
print("Saved:", FINE_TUNED_PATH)

Restored checkpoint from epoch: 27
Trainable parameters: 1158855
Fine-tuning configuration ready.

FINE-TUNING EPOCH 1/10


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 2.0178
Validation Loss: 2.2540
Train Accuracy: 40.38%
Validation Accuracy: 35.82%
Epoch Time: 1.82 minutes
Fine-tuned checkpoint saved!

FINE-TUNING EPOCH 2/10


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

... (21 outputs left out)

Train Loss: 1.8714
Validation Loss: 2.1964
Train Accuracy: 43.75%
Validation Accuracy: 37.19%
Epoch Time: 1.83 minutes
Fine-tuned checkpoint saved!

FINE-TUNING EPOCH 10/10


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 1.8521
Validation Loss: 2.1808
Train Accuracy: 44.66%
Validation Accuracy: 37.79%
Epoch Time: 1.83 minutes
Fine-tuned checkpoint saved!

FINE-TUNING COMPLETED
Best Fine-Tuning Epoch: 10
Best Fine-Tuning Validation Loss: 2.1808
Saved: /kaggle/working/best_finetuned_skin_model.pth


## 11. Final Model Evaluation on the Test Dataset

After fine-tuning, the best fine-tuned checkpoint is loaded for final evaluation.

The model is switched to evaluation mode and tested on the separate test dataset, which was not included in the training or validation split.

### 11.1 Load the Fine-Tuned Model

The checkpoint saved during fine-tuning is restored from:

`/kaggle/working/best_finetuned_skin_model.pth`

The model is set to evaluation mode using `model.eval()` to disable training-specific behavior.

### 11.2 Test DataLoader

A dedicated test DataLoader is created with the following configuration:

| Setting | Value |
|---|---:|
| Batch size | 32 |
| Shuffle | False |
| Workers | 2 |
| Pin memory | Enabled |
| Persistent workers | Enabled when workers are available |

The test dataset uses the evaluation preprocessing pipeline rather than random training augmentations.

### 11.3 Prediction Generation

The model processes test images using:

- `torch.inference_mode()` to disable gradient calculations.
- Automatic mixed precision when enabled.
- `argmax()` to select the class with the highest predicted logit.

Predicted labels and actual labels are collected for metric calculation.

### 11.4 Evaluation Metrics

The following metrics are calculated:

| Metric | Purpose |
|---|---|
| Accuracy | Overall proportion of correctly classified images |
| Macro Precision | Average precision across all classes |
| Macro Recall | Average recall across all classes |
| Macro F1-Score | Average F1-score across all classes |
| Classification Report | Class-wise precision, recall, F1-score and support |
| Confusion Matrix | Class-wise distribution of actual and predicted labels |

Macro-averaged metrics give equal importance to each class, regardless of its number of test images.

### 11.5 Evaluation Output

The notebook displays:

- Total number of evaluated test images.
- Overall test accuracy.
- Macro precision, recall and F1-score.
- Detailed classification report for all 23 classes.
- Confusion matrix dimensions.

**Outcome:** The fine-tuned EfficientNet-B0 model is evaluated across all 23 skin disease categories using overall and class-wise classification metrics.

In [11]:
# ==========================================
# FINAL TEST SET EVALUATION
# ==========================================

# Load best fine-tuned model
checkpoint = torch.load(
    FINE_TUNED_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Create test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

all_predictions = []
all_labels = []

print("=" * 60)
print("EVALUATING ON UNTOUCHED TEST DATA")
print("=" * 60)

with torch.inference_mode():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)

        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_predictions.extend(predictions)
        all_labels.extend(labels.numpy())

# Calculate metrics
test_accuracy = accuracy_score(all_labels, all_predictions)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"Test Images:       {len(all_labels)}")
print(f"Test Accuracy:    {test_accuracy * 100:.2f}%")
print(f"Macro Precision:  {test_precision * 100:.2f}%")
print(f"Macro Recall:     {test_recall * 100:.2f}%")
print(f"Macro F1-Score:   {test_f1 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=test_dataset.classes,
        zero_division=0
    )
)

conf_matrix = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix Shape:", conf_matrix.shape)
print("TEST EVALUATION COMPLETED")

EVALUATING ON UNTOUCHED TEST DATA


Testing:   0%|          | 0/126 [00:00<?, ?it/s]


FINAL TEST RESULTS
Test Images:       4002
Test Accuracy:    38.38%
Macro Precision:  33.69%
Macro Recall:     35.80%
Macro F1-Score:   33.84%

CLASSIFICATION REPORT
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.56      0.56      0.56       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.42      0.44      0.43       288
                                          Atopic Dermatitis Photos       0.39      0.34      0.37       123
                                            Bullous Disease Photos       0.22      0.17      0.19       113
                Cellulitis Impetigo and other Bacterial Infections       0.20      0.11      0.14        73
                                                     Eczema Photos       0.43      0.46      0.44       309
                                      Exanthems and Drug Eruptions       0.2

## 12. Misclassification Analysis

After evaluating the model on the test dataset, this stage examines incorrect predictions to better understand its classification weaknesses.

The confusion matrix generated from the actual and predicted labels is used to identify the most frequent class-level errors.

### 12.1 Top 20 Misclassified Class Pairs

The analysis examines every combination of actual and predicted classes.

For each pair:

- Correct predictions are excluded.
- Incorrect predictions are counted using the confusion matrix.
- Each error is associated with its actual and predicted class names.
- Error pairs are sorted by frequency in descending order.

The notebook displays the 20 most frequent misclassified class pairs.

This helps identify which categories the model most frequently confuses.

### 12.2 Per-Class Accuracy

Accuracy is calculated independently for each class using:

\[
\text{Class Accuracy} =
\frac{\text{Correct Predictions for the Class}}
{\text{Total Actual Samples in the Class}}
\]

The diagonal element of the confusion matrix represents correct predictions for a class, while the corresponding row sum represents all actual samples belonging to that class.

### 12.3 Why This Analysis Matters

Overall accuracy does not reveal how well the model performs on individual categories.

Misclassification analysis helps:

- Identify difficult disease categories.
- Discover frequently confused class pairs.
- Highlight classes with comparatively low accuracy.
- Guide future improvements in data quality, augmentation and model architecture.

**Outcome:** A class-level error analysis that provides additional insight into the strengths and weaknesses of the trained classification model.

In [12]:
# ==========================================
# MISCLASSIFICATION ANALYSIS
# ==========================================

from collections import Counter

confusion = confusion_matrix(all_labels, all_predictions)

print("=" * 65)
print("TOP 20 MISCLASSIFIED CLASS PAIRS")
print("=" * 65)

errors = []

for true_idx in range(NUM_CLASSES):
    for pred_idx in range(NUM_CLASSES):
        if true_idx != pred_idx and confusion[true_idx, pred_idx] > 0:
            errors.append((
                confusion[true_idx, pred_idx],
                test_dataset.classes[true_idx],
                test_dataset.classes[pred_idx]
            ))

errors.sort(reverse=True)

for count, actual, predicted in errors[:20]:
    print(f"{count:4d} | ACTUAL: {actual}")
    print(f"     | PREDICTED: {predicted}\n")

print("=" * 65)
print("PER-CLASS ACCURACY")
print("=" * 65)

for i, class_name in enumerate(test_dataset.classes):
    total = confusion[i].sum()
    correct = confusion[i, i]
    accuracy = correct / total if total else 0

    print(
        f"{class_name[:48]:48s} "
        f"{accuracy * 100:6.2f}%"
    )

TOP 20 MISCLASSIFIED CLASS PAIRS
  48 | ACTUAL: Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions
     | PREDICTED: Seborrheic Keratoses and other Benign Tumors

  40 | ACTUAL: Psoriasis pictures Lichen Planus and related diseases
     | PREDICTED: Tinea Ringworm Candidiasis and other Fungal Infections

  37 | ACTUAL: Psoriasis pictures Lichen Planus and related diseases
     | PREDICTED: Eczema Photos

  32 | ACTUAL: Seborrheic Keratoses and other Benign Tumors
     | PREDICTED: Melanoma Skin Cancer Nevi and Moles

  26 | ACTUAL: Seborrheic Keratoses and other Benign Tumors
     | PREDICTED: Systemic Disease

  26 | ACTUAL: Seborrheic Keratoses and other Benign Tumors
     | PREDICTED: Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions

  23 | ACTUAL: Psoriasis pictures Lichen Planus and related diseases
     | PREDICTED: Seborrheic Keratoses and other Benign Tumors

  22 | ACTUAL: Tinea Ringworm Candidiasis and other Fungal Infections
     | PREDICTED

## 13. Full Backbone Fine-Tuning

After selective fine-tuning, an additional training experiment is performed by unfreezing the complete EfficientNet-B0 backbone.

The best checkpoint from the previous fine-tuning stage is restored before starting this experiment.

### 13.1 Restore the Best Checkpoint

The model loads the checkpoint saved at:

`/kaggle/working/best_finetuned_skin_model.pth`

This ensures that full fine-tuning starts from the best model obtained during selective fine-tuning.

### 13.2 Unfreeze the Complete Backbone

All parameters in `model.features` are made trainable.

Consequently:

- All EfficientNet-B0 feature extraction blocks can be updated.
- The classification head remains trainable.
- The entire network can adapt its learned representations to the target dataset.

Unlike the previous stage, no backbone feature blocks remain frozen.

### 13.3 Differential Learning Rates

AdamW uses separate learning rates for the backbone and classifier.

| Model Component | Learning Rate |
|---|---:|
| Complete EfficientNet-B0 backbone | 0.00001 |
| Classification head | 0.00005 |

A lower learning rate is assigned to the pretrained backbone to support gradual adaptation.

Weight decay is retained to help regularize the model.

### 13.4 Training Configuration

| Setting | Value |
|---|---:|
| Maximum epochs | 5 |
| Early stopping patience | 3 |
| Optimizer | AdamW |
| Scheduler | ReduceLROnPlateau |
| Scheduler factor | 0.5 |
| Scheduler patience | 2 |

### 13.5 Validation and Checkpointing

Training and validation are performed for each epoch.

The scheduler monitors validation loss and adjusts learning rates when necessary.

Whenever validation loss improves, the model checkpoint is saved to:

`/kaggle/working/best_full_finetuned_skin_model.pth`

Training stops early if validation loss does not improve for three consecutive epochs.

### 13.6 Experimental Purpose

Full backbone fine-tuning allows the complete pretrained network to adapt to the skin disease dataset.

However, unfreezing all layers does not guarantee better performance. It can also increase the risk of overfitting or losing useful pretrained representations.

The results should therefore be compared with the previous selective fine-tuning experiment.

**Outcome:** A separately saved full fine-tuned EfficientNet-B0 checkpoint for further evaluation and model comparison.

In [13]:
# ==========================================
# FULL BACKBONE FINE-TUNING
# ==========================================

# Restore best fine-tuned checkpoint
checkpoint = torch.load(
    FINE_TUNED_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

# Unfreeze the complete backbone
for param in model.features.parameters():
    param.requires_grad = True

# Differential learning rates
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.features.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

FULL_TUNE_EPOCHS = 5
FULL_TUNE_PATIENCE = 3

full_best_loss = float("inf")
full_best_epoch = 0
full_wait = 0

FULL_TUNED_PATH = (
    "/kaggle/working/best_full_finetuned_skin_model.pth"
)

print("Restored checkpoint:", checkpoint["epoch"])
print("All EfficientNet backbone layers are trainable.")
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(FULL_TUNE_EPOCHS):

    start = time.time()

    print(f"\nFULL FINE-TUNING EPOCH {epoch + 1}/{FULL_TUNE_EPOCHS}")

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_accuracy = validate_one_epoch(
        model, val_loader, criterion
    )

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
    print(f"Epoch Time: {(time.time() - start) / 60:.2f} minutes")

    if val_loss < full_best_loss:
        full_best_loss = val_loss
        full_best_epoch = epoch + 1
        full_wait = 0

        torch.save({
            "epoch": full_best_epoch,
            "model_state_dict": model.state_dict(),
            "best_val_loss": full_best_loss,
            "class_names": train_classes,
            "image_size": IMAGE_SIZE
        }, FULL_TUNED_PATH)

        print("Best full fine-tuned model saved!")

    else:
        full_wait += 1
        print(f"No improvement: {full_wait}/{FULL_TUNE_PATIENCE}")

    if full_wait >= FULL_TUNE_PATIENCE:
        print("Early stopping triggered.")
        break

print("\nFULL FINE-TUNING COMPLETED")
print("Best Epoch:", full_best_epoch)
print("Best Validation Loss:", round(full_best_loss, 4))
print("Saved:", FULL_TUNED_PATH)

Restored checkpoint: 10
All EfficientNet backbone layers are trainable.
Trainable parameters: 4037011

FULL FINE-TUNING EPOCH 1/5


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 1.8327
Validation Loss: 2.1713
Train Accuracy: 45.30%
Validation Accuracy: 38.47%
Epoch Time: 1.92 minutes
Best full fine-tuned model saved!

FULL FINE-TUNING EPOCH 2/5


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

... (6 outputs left out)

Train Loss: 1.6850
Validation Loss: 2.0841
Train Accuracy: 48.72%
Validation Accuracy: 40.75%
Epoch Time: 1.87 minutes
Best full fine-tuned model saved!

FULL FINE-TUNING EPOCH 5/5


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 1.6573
Validation Loss: 2.0710
Train Accuracy: 49.92%
Validation Accuracy: 41.22%
Epoch Time: 1.87 minutes
Best full fine-tuned model saved!

FULL FINE-TUNING COMPLETED
Best Epoch: 5
Best Validation Loss: 2.071
Saved: /kaggle/working/best_full_finetuned_skin_model.pth


## 14. Model Checkpoint Comparison

This stage compares the three model checkpoints generated during different training strategies.

The comparison includes:

1. Initial Training — classifier head training with a frozen backbone.
2. Partial Fine-Tuning — selective unfreezing of the last two backbone blocks.
3. Full Fine-Tuning — unfreezing of the complete EfficientNet-B0 backbone.

### 14.1 Checkpoint Information

Each checkpoint is loaded individually on the CPU to inspect its saved metadata without requiring GPU memory.

The following information is displayed:

| Attribute | Description |
|---|---|
| Saved Epoch | Epoch at which the best checkpoint was obtained within that training stage |
| Validation Loss | Lowest validation loss recorded for that checkpoint |
| Checkpoint Size | Storage occupied by the saved model file in MB |

### 14.2 Purpose of Comparison

This comparison helps understand how different fine-tuning strategies affect validation performance and model storage requirements.

- Lower validation loss indicates better performance on the validation dataset's loss objective.
- Checkpoint size provides an indication of storage requirements.
- Saved epoch identifies when the best checkpoint was produced.

### 14.3 Model Selection Consideration

Validation loss alone does not determine the final model's overall classification quality.

The final model selection should also consider validation metrics, class-wise performance, test evaluation results and deployment requirements.

**Outcome:** A comparison of the three saved checkpoints to support an informed model-selection decision.

In [14]:
# ==========================================
# CHECKPOINT COMPARISON
# ==========================================

checkpoints_to_compare = {
    "Initial Training": BEST_MODEL_PATH,
    "Partial Fine-Tuning": FINE_TUNED_PATH,
    "Full Fine-Tuning": FULL_TUNED_PATH
}

print("=" * 65)
print("MODEL CHECKPOINT COMPARISON")
print("=" * 65)

for name, path in checkpoints_to_compare.items():

    checkpoint_data = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{name}")
    print("-" * 45)
    print("Saved Epoch:", checkpoint_data["epoch"])
    print("Validation Loss:",
          round(checkpoint_data["best_val_loss"], 4))
    print("Checkpoint Size:",
          f"{Path(path).stat().st_size / (1024 * 1024):.2f} MB")

print("\nCHECKPOINT COMPARISON COMPLETED")

MODEL CHECKPOINT COMPARISON

Initial Training
---------------------------------------------
Saved Epoch: 27
Validation Loss: 2.2778
Checkpoint Size: 15.92 MB

Partial Fine-Tuning
---------------------------------------------
Saved Epoch: 10
Validation Loss: 2.1808
Checkpoint Size: 15.69 MB

Full Fine-Tuning
---------------------------------------------
Saved Epoch: 5
Validation Loss: 2.071
Checkpoint Size: 15.69 MB

CHECKPOINT COMPARISON COMPLETED


## 15. Final Evaluation of the Full Fine-Tuned Model

This stage evaluates the model produced by full backbone fine-tuning.

The checkpoint is loaded from:

`/kaggle/working/best_full_finetuned_skin_model.pth`

The model is switched to evaluation mode before generating predictions on the test dataset.

### 15.1 Prediction Generation

The model processes the test images using:

- `torch.inference_mode()` to disable gradient calculations.
- Automatic mixed precision when enabled.
- The existing test DataLoader.
- `argmax()` to select the predicted class from the model outputs.

Actual labels and predicted labels are collected for evaluation.

### 15.2 Evaluation Metrics

The following metrics are calculated:

| Metric | Description |
|---|---|
| Test Accuracy | Overall percentage of correctly classified images |
| Macro Precision | Average precision across all 23 classes |
| Macro Recall | Average recall across all 23 classes |
| Macro F1-Score | Average F1-score across all 23 classes |
| Classification Report | Detailed class-wise precision, recall, F1-score and support |

Macro averaging gives equal weight to every class, providing additional insight when class distributions are imbalanced.

### 15.3 Comparison with Previous Fine-Tuning

The results from this evaluation can be compared with the partial fine-tuning results.

This helps determine whether unfreezing the complete EfficientNet-B0 backbone improved classification performance or introduced overfitting.

### 15.4 Final Evaluation Output

The notebook displays:

- Total test images evaluated.
- Overall test accuracy.
- Macro precision, recall and F1-score.
- Detailed classification metrics for all 23 categories.

**Outcome:** Performance measurements for the full fine-tuned EfficientNet-B0 model, supporting the final checkpoint selection process.

In [15]:
# ==========================================
# FINAL EVALUATION - BEST MODEL
# ==========================================

checkpoint = torch.load(
    FULL_TUNED_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_predictions = []
all_labels = []

print("=" * 60)
print("FINAL MODEL TEST EVALUATION")
print("=" * 60)

with torch.inference_mode():
    for images, labels in tqdm(test_loader, desc="Final Testing"):

        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)

        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_predictions.extend(predictions)
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(all_labels, all_predictions)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )
)

print("\n" + "=" * 60)
print("FINAL MODEL PERFORMANCE")
print("=" * 60)

print(f"Test Images:      {len(all_labels)}")
print(f"Test Accuracy:   {test_accuracy * 100:.2f}%")
print(f"Macro Precision: {test_precision * 100:.2f}%")
print(f"Macro Recall:    {test_recall * 100:.2f}%")
print(f"Macro F1-Score:  {test_f1 * 100:.2f}%")

print("\nCLASSIFICATION REPORT")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=test_dataset.classes,
    zero_division=0
))

print("FINAL EVALUATION COMPLETED")

FINAL MODEL TEST EVALUATION


Final Testing:   0%|          | 0/126 [00:00<?, ?it/s]


FINAL MODEL PERFORMANCE
Test Images:      4002
Test Accuracy:   42.05%
Macro Precision: 37.44%
Macro Recall:    39.74%
Macro F1-Score:  37.80%

CLASSIFICATION REPORT
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.67      0.57      0.62       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.45      0.52      0.48       288
                                          Atopic Dermatitis Photos       0.34      0.40      0.36       123
                                            Bullous Disease Photos       0.26      0.25      0.26       113
                Cellulitis Impetigo and other Bacterial Infections       0.22      0.14      0.17        73
                                                     Eczema Photos       0.46      0.50      0.48       309
                                      Exanthems and Drug Eruptions       0.3

## 16. Balanced Data Sampling

The skin disease dataset may contain different numbers of images across its 23 classes.

When a model is trained on an imbalanced dataset, it may become biased toward classes with more training examples.

To address this issue, a `WeightedRandomSampler` is introduced to improve the representation of minority classes during training.

### 16.1 Training Class Distribution

Class frequencies are calculated using only the training split.

Validation and test samples are excluded from the sampling-weight calculation to prevent data leakage.

### 16.2 Inverse-Square-Root Sampling Weights

A moderate inverse-square-root weighting strategy is used:

\[
w_c = \frac{1}{\sqrt{n_c}}
\]

Where:

- \(w_c\) represents the sampling weight of class \(c\).
- \(n_c\) represents the number of training samples in class \(c\).

This gives minority classes a higher sampling weight while avoiding the stronger correction produced by full inverse-frequency weighting.

### 16.3 Per-Image Sampling Weights

Each training image receives the sampling weight associated with its class.

The resulting weights are passed to `WeightedRandomSampler`.

The sampler configuration uses:

| Setting | Value |
|---|---|
| Number of samples | Training dataset size |
| Replacement | Enabled |
| Sampling strategy | Weighted random sampling |

Replacement sampling allows an image to be selected more than once during an epoch, while some other images may not be selected.

Therefore, this approach encourages more balanced class exposure but does not guarantee exactly equal class counts in every epoch.

### 16.4 Balanced Training DataLoader

A new DataLoader named `balanced_train_loader` is created using the weighted sampler.

The original training DataLoader remains available, allowing experiments with standard and balanced sampling strategies.

Validation and test DataLoaders are not modified.

### 16.5 Configuration Verification

The cell displays:

- Number of classes.
- Total training images.
- Number of balanced batches.
- Minimum and maximum class counts.
- Sampler type.
- Replacement sampling configuration.

**Outcome:** A weighted training sampler is prepared to reduce the influence of class imbalance during subsequent model training.

In [16]:
# ==========================================
# BALANCED DATA SAMPLING
# ==========================================

from torch.utils.data import WeightedRandomSampler

# Class counts from training split only
train_targets = np.array(train_base.targets)[train_indices]

class_sample_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES
)

# Moderate inverse-square-root sampling weights
class_sampling_weights = 1.0 / np.sqrt(class_sample_counts)

# Assign a weight to every training image
sample_weights = class_sampling_weights[train_targets]

balanced_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

# New balanced training loader
balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

print("=" * 55)
print("BALANCED SAMPLING CONFIGURATION")
print("=" * 55)

print("Number of classes:", NUM_CLASSES)
print("Training images:", len(train_dataset))
print("Balanced batches:", len(balanced_train_loader))
print("Minimum class count:", class_sample_counts.min())
print("Maximum class count:", class_sample_counts.max())
print("Sampler:", type(balanced_sampler).__name__)
print("Replacement sampling: Enabled")

print("\nBALANCED SAMPLER READY")

BALANCED SAMPLING CONFIGURATION
Number of classes: 23
Training images: 13223
Balanced batches: 414
Minimum class count: 180
Maximum class count: 1194
Sampler: WeightedRandomSampler
Replacement sampling: Enabled

BALANCED SAMPLER READY


## 17. Balanced Training Loss and Hyperparameter Configuration

After preparing the weighted training sampler, this stage configures the loss function and learning rates for the balanced training experiment.

### 17.1 Loss Function

Standard `CrossEntropyLoss` is used for this experiment.

Unlike the earlier training stage, no explicit class weights are passed to the loss function.

Class imbalance is addressed through the `WeightedRandomSampler`, which adjusts the probability of selecting individual training images.

This keeps sampling-based balancing separate from loss-based class weighting.

### 17.2 Differential Learning Rates

Separate learning rates are configured for the backbone and classification head.

| Model Component | Learning Rate |
|---|---:|
| EfficientNet-B0 backbone | 0.000005 |
| Classification head | 0.00005 |

The lower backbone learning rate supports gradual updates to pretrained feature representations, while the classifier receives a comparatively higher learning rate.

### 17.3 Balanced Training Configuration

The experiment combines:

- `WeightedRandomSampler` for weighted image selection.
- Standard CrossEntropyLoss.
- Differential learning rates for the backbone and classifier.
- The previously prepared balanced training DataLoader.

### 17.4 Configuration Verification

The cell displays the selected loss function, learning rates, sampler type and number of training batches.

**Outcome:** The balanced training experiment is configured and ready for model training.

In [17]:
# ==========================================
# BALANCED TRAINING LOSS
# ==========================================

balanced_criterion = nn.CrossEntropyLoss()

BALANCED_HEAD_LR = 5e-5
BALANCED_BACKBONE_LR = 5e-6

print("=" * 55)
print("BALANCED TRAINING CONFIGURATION")
print("=" * 55)

print("Loss Function:", type(balanced_criterion).__name__)
print("Backbone Learning Rate:", BALANCED_BACKBONE_LR)
print("Classifier Learning Rate:", BALANCED_HEAD_LR)
print("Sampler: WeightedRandomSampler")
print("Training batches:", len(balanced_train_loader))

print("\nBALANCED TRAINING CONFIGURATION READY")

BALANCED TRAINING CONFIGURATION
Loss Function: CrossEntropyLoss
Backbone Learning Rate: 5e-06
Classifier Learning Rate: 5e-05
Sampler: WeightedRandomSampler
Training batches: 414

BALANCED TRAINING CONFIGURATION READY


## 18. Balanced Model Training

This stage trains an additional model using weighted random sampling to reduce the effect of class imbalance.

Training starts from the best checkpoint obtained during full backbone fine-tuning.

### 18.1 Restore the Previous Best Model

The checkpoint is restored from:

`/kaggle/working/best_full_finetuned_skin_model.pth`

All model parameters are enabled for training, including the complete EfficientNet-B0 backbone and classification head.

### 18.2 Optimizer Configuration

AdamW is configured with separate learning rates:

| Model Component | Learning Rate |
|---|---:|
| EfficientNet-B0 backbone | 0.000005 |
| Classification head | 0.00005 |

Weight decay is retained from the earlier configuration.

### 18.3 Balanced Training Strategy

The model is trained using:

- `balanced_train_loader` with `WeightedRandomSampler`.
- Standard CrossEntropyLoss.
- The complete trainable EfficientNet-B0 architecture.
- Differential learning rates for the backbone and classifier.

This experiment uses sampling-based balancing rather than explicit class-weighted loss.

### 18.4 Training Configuration

| Setting | Value |
|---|---:|
| Maximum epochs | 8 |
| Early stopping patience | 3 |
| Optimizer | AdamW |
| Scheduler | ReduceLROnPlateau |
| Scheduler factor | 0.5 |
| Scheduler patience | 2 |

### 18.5 Validation and Checkpointing

After each training epoch, the model is evaluated using the validation DataLoader.

The scheduler monitors validation loss.

Whenever validation loss improves, the model checkpoint is saved to:

`/kaggle/working/best_balanced_skin_model.pth`

Training stops early if validation loss does not improve for three consecutive epochs.

### 18.6 Experimental Purpose

The balanced training experiment investigates whether increasing the sampling probability of minority-class images improves the model's ability to recognize less-represented skin disease categories.

Its performance should be compared against the previous full fine-tuned model using consistent evaluation metrics.

**Outcome:** A separately saved balanced EfficientNet-B0 checkpoint for subsequent evaluation and model comparison.

In [18]:
# ==========================================
# BALANCED TRAINING PHASE
# ==========================================

# Restore best previous model
checkpoint = torch.load(
    FULL_TUNED_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

# Ensure complete model is trainable
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.features.parameters(),
            "lr": BALANCED_BACKBONE_LR
        },
        {
            "params": model.classifier.parameters(),
            "lr": BALANCED_HEAD_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

BALANCED_EPOCHS = 8
BALANCED_PATIENCE = 3

balanced_best_loss = float("inf")
balanced_best_epoch = 0
balanced_wait = 0

BALANCED_MODEL_PATH = (
    "/kaggle/working/best_balanced_skin_model.pth"
)

print("Starting from best full fine-tuned checkpoint.")
print("Balanced training batches:", len(balanced_train_loader))

for epoch in range(BALANCED_EPOCHS):

    start = time.time()

    print(f"\nBALANCED EPOCH {epoch + 1}/{BALANCED_EPOCHS}")

    train_loss, train_accuracy = train_one_epoch(
        model,
        balanced_train_loader,
        balanced_criterion,
        optimizer
    )

    val_loss, val_accuracy = validate_one_epoch(
        model,
        val_loader,
        balanced_criterion
    )

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
    print(f"Epoch Time: {(time.time() - start) / 60:.2f} minutes")

    if val_loss < balanced_best_loss:
        balanced_best_loss = val_loss
        balanced_best_epoch = epoch + 1
        balanced_wait = 0

        torch.save({
            "epoch": balanced_best_epoch,
            "model_state_dict": model.state_dict(),
            "best_val_loss": balanced_best_loss,
            "class_names": train_classes,
            "image_size": IMAGE_SIZE
        }, BALANCED_MODEL_PATH)

        print("Best balanced model saved!")

    else:
        balanced_wait += 1
        print(f"No improvement: {balanced_wait}/{BALANCED_PATIENCE}")

    if balanced_wait >= BALANCED_PATIENCE:
        print("Balanced training early stopping triggered.")
        break

print("\nBALANCED TRAINING COMPLETED")
print("Best Epoch:", balanced_best_epoch)
print("Best Validation Loss:", round(balanced_best_loss, 4))
print("Saved:", BALANCED_MODEL_PATH)

Starting from best full fine-tuned checkpoint.
Balanced training batches: 414

BALANCED EPOCH 1/8


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 1.5857
Validation Loss: 1.9968
Train Accuracy: 52.02%
Validation Accuracy: 41.60%
Epoch Time: 1.83 minutes
Best balanced model saved!

BALANCED EPOCH 2/8


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

... (15 outputs left out)

Train Loss: 1.4656
Validation Loss: 1.9538
Train Accuracy: 55.49%
Validation Accuracy: 43.36%
Epoch Time: 1.81 minutes
No improvement: 1/3

BALANCED EPOCH 8/8


Training:   0%|          | 0/414 [00:00<?, ?it/s]

Validation:   0%|          | 0/73 [00:00<?, ?it/s]

Train Loss: 1.4694
Validation Loss: 1.9367
Train Accuracy: 55.50%
Validation Accuracy: 43.44%
Epoch Time: 1.80 minutes
Best balanced model saved!

BALANCED TRAINING COMPLETED
Best Epoch: 8
Best Validation Loss: 1.9367
Saved: /kaggle/working/best_balanced_skin_model.pth


## 19. Validation-Based Checkpoint Comparison

This stage compares the four model checkpoints produced by different training strategies.

The purpose is to examine how each training approach affects validation loss, the epoch at which the best checkpoint was obtained, and model storage requirements.

### 19.1 Training Strategies Compared

| Checkpoint | Training Strategy |
|---|---|
| Initial | Frozen backbone with classifier head training |
| Partial Fine-Tuning | Last two backbone blocks unfrozen |
| Full Fine-Tuning | Complete backbone unfrozen |
| Balanced Training | Full model training with weighted random sampling |

### 19.2 Comparison Metrics

Each saved checkpoint is loaded on the CPU, and the following information is extracted:

- **Best Epoch:** Epoch at which the lowest validation loss was recorded within that training stage.
- **Validation Loss:** Best validation loss associated with the checkpoint.
- **File Size:** Storage occupied by the checkpoint in megabytes.

### 19.3 Model Selection

Validation loss provides a useful basis for comparing the training experiments.

However, lower validation loss does not automatically guarantee better class-wise classification performance.

The final selection should also consider macro precision, macro recall, macro F1-score, per-class accuracy and confusion matrix results.

### 19.4 Experimental Purpose

This comparison helps determine whether balanced sampling provides an advantage over standard training and fine-tuning strategies.

**Outcome:** A four-checkpoint validation comparison to support the selection of the final model.

In [19]:
# ==========================================
# VALIDATION CHECKPOINT COMPARISON
# ==========================================

comparison_paths = {
    "Initial": BEST_MODEL_PATH,
    "Partial Fine-Tuning": FINE_TUNED_PATH,
    "Full Fine-Tuning": FULL_TUNED_PATH,
    "Balanced Training": BALANCED_MODEL_PATH
}

print("=" * 65)
print("VALIDATION CHECKPOINT COMPARISON")
print("=" * 65)

for name, path in comparison_paths.items():

    data = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{name}")
    print("-" * 45)
    print("Best Epoch:", data["epoch"])
    print("Validation Loss:", round(data["best_val_loss"], 4))
    print(
        "File Size:",
        f"{Path(path).stat().st_size / (1024 * 1024):.2f} MB"
    )

print("\nCOMPARISON COMPLETED")

VALIDATION CHECKPOINT COMPARISON

Initial
---------------------------------------------
Best Epoch: 27
Validation Loss: 2.2778
File Size: 15.92 MB

Partial Fine-Tuning
---------------------------------------------
Best Epoch: 10
Validation Loss: 2.1808
File Size: 15.69 MB

Full Fine-Tuning
---------------------------------------------
Best Epoch: 5
Validation Loss: 2.071
File Size: 15.69 MB

Balanced Training
---------------------------------------------
Best Epoch: 8
Validation Loss: 1.9367
File Size: 15.69 MB

COMPARISON COMPLETED


## 20. Validation Performance Comparison

This stage performs a detailed comparison between the Full Fine-Tuned model and the Balanced Model using the validation dataset.

Unlike the previous checkpoint comparison, which focused on saved validation loss and file size, this experiment calculates classification metrics by generating fresh predictions on the validation dataset.

### 20.1 Models Compared

Two checkpoints are evaluated:

- **Full Fine-Tuned:** EfficientNet-B0 trained with the complete backbone unfrozen.
- **Balanced Model:** EfficientNet-B0 trained using weighted random sampling.

### 20.2 Evaluation Procedure

For each checkpoint:

1. Load the saved model weights.
2. Switch the model to evaluation mode.
3. Generate predictions using the validation DataLoader.
4. Collect actual and predicted class labels.
5. Calculate the classification metrics.

Inference is performed using `torch.inference_mode()` and automatic mixed precision when enabled.

### 20.3 Evaluation Metrics

| Metric | Description |
|---|---|
| Accuracy | Overall proportion of correct predictions |
| Macro Precision | Average precision across all classes |
| Macro Recall | Average recall across all classes |
| Macro F1-Score | Average F1-score across all classes |

Macro-averaged metrics give equal importance to each of the 23 classes.

### 20.4 Comparison Objective

The comparison helps determine whether balanced sampling improves classification performance, particularly for less-represented classes.

Special attention should be given to macro recall and macro F1-score rather than relying only on overall accuracy.

### 20.5 Results

The notebook prints the four evaluation metrics for both models, making their validation performance directly comparable.

**Outcome:** A metric-based validation comparison to help identify the stronger candidate for final model selection.

In [20]:
# ==========================================
# VALIDATION METRICS COMPARISON
# ==========================================

validation_models = {
    "Full Fine-Tuned": FULL_TUNED_PATH,
    "Balanced Model": BALANCED_MODEL_PATH
}

validation_results = {}

for model_name, checkpoint_path in validation_models.items():

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    predictions_list = []
    labels_list = []

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Evaluating {model_name}"
        ):

            images = images.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                outputs = model(images)

            predictions_list.extend(
                outputs.argmax(dim=1).cpu().numpy()
            )
            labels_list.extend(labels.numpy())

    accuracy = accuracy_score(labels_list, predictions_list)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_list,
        predictions_list,
        average="macro",
        zero_division=0
    )

    validation_results[model_name] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("\n" + "=" * 65)
print("VALIDATION METRICS COMPARISON")
print("=" * 65)

for name, metrics in validation_results.items():
    print(f"\n{name}")
    print("-" * 40)
    print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
    print(f"Precision: {metrics['precision'] * 100:.2f}%")
    print(f"Recall:    {metrics['recall'] * 100:.2f}%")
    print(f"Macro F1:  {metrics['f1'] * 100:.2f}%")

print("\nVALIDATION COMPARISON COMPLETED")

Evaluating Full Fine-Tuned:   0%|          | 0/73 [00:00<?, ?it/s]

Evaluating Balanced Model:   0%|          | 0/73 [00:00<?, ?it/s]


VALIDATION METRICS COMPARISON

Full Fine-Tuned
----------------------------------------
Accuracy:  41.22%
Precision: 38.08%
Recall:    39.03%
Macro F1:  37.91%

Balanced Model
----------------------------------------
Accuracy:  43.44%
Precision: 40.25%
Recall:    41.20%
Macro F1:  40.08%

VALIDATION COMPARISON COMPLETED


## 21. Best Validation Model Selection

After comparing the validation performance of the Full Fine-Tuned and Balanced models, the Balanced Model is selected as the current best candidate.

### 21.1 Selected Model

| Attribute | Selected Configuration |
|---|---|
| Architecture | EfficientNet-B0 |
| Training Strategy | Balanced Sampling + Full Fine-Tuning |
| Checkpoint | `best_balanced_skin_model.pth` |
| Validation Accuracy | 43.44% |
| Validation Macro F1-Score | 40.08% |

### 21.2 Restore Checkpoint Information

The selected checkpoint is loaded on the CPU to inspect its saved metadata.

The notebook displays:

- Model architecture.
- Training strategy.
- Checkpoint location.
- Best training epoch.
- Best validation loss.
- Recorded validation accuracy.
- Recorded validation macro F1-score.

### 21.3 Model Selection Rationale

Balanced sampling was introduced to improve the representation of minority classes during training.

The selected candidate achieved the reported validation metrics and is carried forward for final test evaluation and subsequent deployment preparation.

### 21.4 Important Distinction

This cell selects the candidate checkpoint but does not load its weights into the model for inference.

The actual checkpoint loading and final evaluation are performed in the subsequent evaluation stage.

**Outcome:** The balanced EfficientNet-B0 checkpoint is designated as the selected candidate for final evaluation.

In [21]:
# ==========================================
# SELECT BEST VALIDATION MODEL
# ==========================================

BEST_CANDIDATE_PATH = BALANCED_MODEL_PATH

best_candidate_checkpoint = torch.load(
    BEST_CANDIDATE_PATH,
    map_location="cpu",
    weights_only=True
)

print("=" * 60)
print("BEST MODEL SELECTED")
print("=" * 60)

print("Model: EfficientNet-B0")
print("Training Strategy: Balanced Sampling + Full Fine-Tuning")
print("Checkpoint:", BEST_CANDIDATE_PATH)
print("Best Epoch:", best_candidate_checkpoint["epoch"])
print("Validation Loss:",
      best_candidate_checkpoint["best_val_loss"])
print("Validation Accuracy: 43.44%")
print("Validation Macro F1: 40.08%")

print("\nBEST CANDIDATE READY FOR FINAL TEST")

BEST MODEL SELECTED
Model: EfficientNet-B0
Training Strategy: Balanced Sampling + Full Fine-Tuning
Checkpoint: /kaggle/working/best_balanced_skin_model.pth
Best Epoch: 8
Validation Loss: 1.936710828578462
Validation Accuracy: 43.44%
Validation Macro F1: 40.08%

BEST CANDIDATE READY FOR FINAL TEST


## 22. Final Test Evaluation — Selected Balanced Model

After selecting the Balanced EfficientNet-B0 checkpoint using validation performance, the selected model is evaluated on the separate test dataset.

This stage provides the final classification metrics for the selected model.

### 22.1 Load the Selected Checkpoint

The selected checkpoint is loaded from:

`BEST_CANDIDATE_PATH`

The model weights are restored and the model is switched to evaluation mode.

### 22.2 Test DataLoader

A dedicated test DataLoader is configured with:

- Batch size of 32.
- Sequential data loading without shuffling.
- Two worker processes.
- Pinned memory.
- Persistent workers when enabled.

The test dataset uses the evaluation preprocessing pipeline.

### 22.3 Inference Process

Predictions are generated using:

- `torch.inference_mode()` to disable gradient calculations.
- Automatic mixed precision when enabled.
- `argmax()` to select the predicted class.

Actual labels and predicted labels are collected for metric calculation.

### 22.4 Overall Evaluation Metrics

The following macro-averaged metrics are calculated:

| Metric | Description |
|---|---|
| Test Accuracy | Overall percentage of correct predictions |
| Macro Precision | Average precision across all 23 classes |
| Macro Recall | Average recall across all 23 classes |
| Macro F1-Score | Average F1-score across all 23 classes |

Macro averaging assigns equal importance to every class.

### 22.5 Per-Class Classification Report

The classification report provides detailed performance for each of the 23 skin disease categories.

It includes:

- Precision
- Recall
- F1-score
- Support (number of actual test samples)

This helps identify categories where the model performs well and categories that require further improvement.

### 22.6 Final Evaluation Objective

The results provide an overall and class-wise assessment of the selected Balanced EfficientNet-B0 model.

**Outcome:** Final test performance metrics and a detailed classification report for the selected model.

In [22]:
# ==========================================
# FINAL TEST EVALUATION
# ==========================================

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# Load selected checkpoint
checkpoint = torch.load(
    BEST_CANDIDATE_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

test_predictions = []
test_labels = []

# Inference
with torch.inference_mode():

    for images, labels in tqdm(
        test_loader,
        desc="Final Test Evaluation"
    ):

        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)

        predictions = outputs.argmax(dim=1)

        test_predictions.extend(predictions.cpu().numpy())
        test_labels.extend(labels.numpy())

# Metrics
test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        test_labels,
        test_predictions,
        average="macro",
        zero_division=0
    )
)

print("\n" + "=" * 65)
print("FINAL TEST RESULTS — BALANCED MODEL")
print("=" * 65)

print(f"Test Accuracy:  {test_accuracy * 100:.2f}%")
print(f"Macro Precision: {test_precision * 100:.2f}%")
print(f"Macro Recall:    {test_recall * 100:.2f}%")
print(f"Macro F1 Score:  {test_f1 * 100:.2f}%")

print("\n" + "=" * 65)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 65)

print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=train_classes,
        zero_division=0,
        digits=4
    )
)

Final Test Evaluation:   0%|          | 0/126 [00:00<?, ?it/s]


FINAL TEST RESULTS — BALANCED MODEL
Test Accuracy:  43.70%
Macro Precision: 39.52%
Macro Recall:    41.56%
Macro F1 Score:  39.85%

PER-CLASS CLASSIFICATION REPORT
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos     0.6417    0.6314    0.6365       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions     0.4755    0.5382    0.5049       288
                                          Atopic Dermatitis Photos     0.4000    0.3902    0.3951       123
                                            Bullous Disease Photos     0.2718    0.2478    0.2593       113
                Cellulitis Impetigo and other Bacterial Infections     0.2895    0.1507    0.1982        73
                                                     Eczema Photos     0.4467    0.4887    0.4668       309
                                      Exanthems and Drug Eruptions     0.3118 

## 23. Export Developer-Ready Model Package

After selecting and evaluating the Balanced EfficientNet-B0 model, the trained model is packaged into a reusable format for developers.

The objective is to make the model usable outside the training notebook without requiring the complete training pipeline.

### 23.1 Package Directory

All exported files are organized inside:

`/kaggle/working/skin_disease_model/`

### 23.2 Model Weights

The selected model's state dictionary is exported as:

`model.pth`

The weights are stored on the CPU, making the exported checkpoint independent of the training device.

### 23.3 Model Metadata

A `metadata.json` file stores important model information, including:

- Model name and architecture.
- Classification task and number of classes.
- All 23 class names.
- Input image dimensions.
- Image preprocessing configuration.
- Normalization parameters.
- Training strategy.
- Best training epoch.
- Final test evaluation metrics.
- Medical-use limitations.

This metadata allows developers to understand the model configuration without inspecting the original training notebook.

### 23.4 Reusable Inference Script

The package includes:

`inference.py`

This script provides a reusable prediction interface.

Its functionality includes:

1. Loading the model architecture.
2. Restoring the exported model weights.
3. Applying the required image preprocessing.
4. Accepting an input image path.
5. Generating class probabilities using softmax.
6. Returning the predicted class and confidence.
7. Returning the top-k predictions.
8. Including the model-use disclaimer.

The model is loaded lazily and reused for subsequent predictions within the same Python process.

Example usage:

```bash
python inference.py sample.jpg
```

For the top five predictions:

```bash
python inference.py sample.jpg --top-k 5
```

### 23.5 Dependencies

A `requirements.txt` file is included with the required Python packages:

- PyTorch
- Torchvision
- Pillow

### 23.6 ZIP Packaging

The complete model directory is compressed into:

`skin_disease_model_package.zip`

This makes the package easier to distribute, upload and share with other developers.

### 23.7 Export Verification

The notebook verifies the exported files and displays:

- Individual file sizes.
- ZIP archive size.
- Number of supported classes.
- Test accuracy.
- Test macro F1-score.

**Outcome:** A developer-oriented model package containing trained weights, metadata, an inference script and dependency information.

In [23]:
# ==========================================
# EXPORT DEVELOPER-READY MODEL PACKAGE
# ==========================================

import json
import shutil
from pathlib import Path

# Package directory
PACKAGE_DIR = Path("/kaggle/working/skin_disease_model")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save selected model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.cpu()
model.eval()

MODEL_EXPORT_PATH = PACKAGE_DIR / "model.pth"

torch.save(
    {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    },
    MODEL_EXPORT_PATH
)

# 2. Save metadata
metadata = {
    "model_name": "Skin Disease Image Classifier",
    "architecture": "EfficientNet-B0",
    "task": "23-class skin disease image classification",
    "num_classes": NUM_CLASSES,
    "class_names": list(train_classes),
    "input_size": [224, 224],
    "preprocessing": {
        "resize": 256,
        "center_crop": 224,
        "normalization_mean": [0.485, 0.456, 0.406],
        "normalization_std": [0.229, 0.224, 0.225]
    },
    "training_strategy": "Balanced Sampling + Full Fine-Tuning",
    "best_epoch": int(checkpoint["epoch"]),
    "evaluation": {
        "test_accuracy": round(float(test_accuracy), 4),
        "macro_precision": round(float(test_precision), 4),
        "macro_recall": round(float(test_recall), 4),
        "macro_f1": round(float(test_f1), 4)
    },
    "limitations": (
        "Research and educational prototype only. "
        "Not intended for medical diagnosis or treatment decisions."
    )
}

with open(PACKAGE_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# 3. Create reusable inference script
inference_code = r'''
import json
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms


PACKAGE_DIR = Path(__file__).resolve().parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


with open(PACKAGE_DIR / "metadata.json", "r") as f:
    metadata = json.load(f)


def load_model():
    model = models.efficientnet_b0(weights=None)

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(
            model.classifier[1].in_features,
            metadata["num_classes"]
        )
    )

    state_dict = torch.load(
        PACKAGE_DIR / "model.pth",
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()

    return model


preprocess = transforms.Compose([
    transforms.Resize(metadata["preprocessing"]["resize"]),
    transforms.CenterCrop(metadata["input_size"]),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=metadata["preprocessing"]["normalization_mean"],
        std=metadata["preprocessing"]["normalization_std"]
    )
])


_model = None


def predict(image_path, top_k=3):
    global _model

    if _model is None:
        _model = load_model()

    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        logits = _model(image_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    k = min(top_k, len(metadata["class_names"]))
    scores, indices = torch.topk(probabilities, k=k)

    results = []

    for score, index in zip(scores.cpu().tolist(),
                            indices.cpu().tolist()):
        results.append({
            "class_name": metadata["class_names"][index],
            "confidence": round(score, 4)
        })

    return {
        "predicted_class": results[0]["class_name"],
        "confidence": results[0]["confidence"],
        "top_predictions": results,
        "disclaimer": metadata["limitations"]
    }


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("image", help="Path to input image")
    parser.add_argument("--top-k", type=int, default=3)
    args = parser.parse_args()

    print(predict(args.image, top_k=args.top_k))
'''

with open(PACKAGE_DIR / "inference.py", "w") as f:
    f.write(inference_code)

# 4. Requirements
requirements = """torch
torchvision
Pillow
"""

with open(PACKAGE_DIR / "requirements.txt", "w") as f:
    f.write(requirements)

# 5. Create ZIP package
ZIP_PATH = shutil.make_archive(
    "/kaggle/working/skin_disease_model_package",
    "zip",
    root_dir=PACKAGE_DIR
)

# 6. Verify exported files
print("=" * 60)
print("MODEL PACKAGE EXPORTED SUCCESSFULLY")
print("=" * 60)

for file_path in sorted(PACKAGE_DIR.iterdir()):
    print(f"{file_path.name:<25} {file_path.stat().st_size / (1024**2):.2f} MB")

print("\nZIP PACKAGE:", ZIP_PATH)
print("ZIP SIZE:", round(Path(ZIP_PATH).stat().st_size / (1024**2), 2), "MB")

print("\nClasses:", len(metadata["class_names"]))
print("Test Accuracy:", metadata["evaluation"]["test_accuracy"])
print("Test Macro F1:", metadata["evaluation"]["macro_f1"])

MODEL PACKAGE EXPORTED SUCCESSFULLY
inference.py              0.00 MB
metadata.json             0.00 MB
model.pth                 15.65 MB
requirements.txt          0.00 MB

ZIP PACKAGE: /kaggle/working/skin_disease_model_package.zip
ZIP SIZE: 14.48 MB

Classes: 23
Test Accuracy: 0.437
Test Macro F1: 0.3985


## 24. Verify the Exported Inference Package

After exporting the model package, an inference verification test is performed to confirm that the exported files can be loaded and used independently of the training pipeline.

### 24.1 Import the Exported Inference Module

The `inference.py` file is dynamically imported from the exported package directory using Python's import utilities.

This verifies that the inference script can be loaded as a standalone module.

### 24.2 Select a Test Image

One real image is selected from the test dataset.

The image's actual class label is retained for comparison with the model prediction.

### 24.3 Run Prediction

The exported inference module's `predict()` function is called with:

- The selected image path.
- `top_k=3` to return the three highest-scoring classes.

The inference script independently handles model loading, preprocessing and prediction.

### 24.4 Inspect Prediction Results

The output displays:

- Input image filename.
- Actual class.
- Predicted class.
- Prediction confidence.
- Top three class predictions and their confidence scores.
- Model-use disclaimer.

### 24.5 Verification Objective

This test checks that:

1. The exported inference module imports successfully.
2. The exported model weights load correctly.
3. Image preprocessing runs successfully.
4. Prediction results are returned in the expected format.
5. The disclaimer is included in the output.

**Outcome:** A successful run confirms that the exported inference package can perform a prediction on a real test image.

**Note:** This is a single-image inference smoke test. It verifies package functionality, not overall model accuracy or prediction reliability.

In [24]:
# ==========================================
# VERIFY EXPORTED INFERENCE PACKAGE
# ==========================================

import sys
import importlib.util

PACKAGE_DIR = Path("/kaggle/working/skin_disease_model")

# Load inference.py from exported package
inference_path = PACKAGE_DIR / "inference.py"

spec = importlib.util.spec_from_file_location(
    "skin_disease_inference",
    inference_path
)

inference_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = inference_module
spec.loader.exec_module(inference_module)

# Select one real image from test dataset
sample_image_path, actual_label = test_dataset.samples[0]

# Run prediction using exported inference code
prediction = inference_module.predict(
    sample_image_path,
    top_k=3
)

print("=" * 60)
print("EXPORTED INFERENCE TEST")
print("=" * 60)

print("Image:", Path(sample_image_path).name)
print("Actual Class:", train_classes[actual_label])

print("\nPredicted Class:", prediction["predicted_class"])
print("Confidence:", prediction["confidence"])

print("\nTop 3 Predictions:")
for item in prediction["top_predictions"]:
    print(
        f"{item['class_name']:<65} "
        f"{item['confidence']:.4f}"
    )

print("\nDisclaimer:", prediction["disclaimer"])
print("\nINFERENCE PACKAGE VERIFICATION COMPLETE")

EXPORTED INFERENCE TEST
Image: 07PerioralDermEye.jpg
Actual Class: Acne and Rosacea Photos

Predicted Class: Acne and Rosacea Photos
Confidence: 0.381

Top 3 Predictions:
Acne and Rosacea Photos                                           0.3810
Tinea Ringworm Candidiasis and other Fungal Infections            0.1584
Vascular Tumors                                                   0.0766

Disclaimer: Research and educational prototype only. Not intended for medical diagnosis or treatment decisions.

INFERENCE PACKAGE VERIFICATION COMPLETE


# Final Conclusion

## Project Summary

This project developed a deep learning-based **23-class Skin Disease Image Classification** system using PyTorch and the EfficientNet-B0 architecture.

The workflow covered dataset inspection, image preprocessing, transfer learning, selective fine-tuning, full fine-tuning, class-imbalance handling, model evaluation, and developer-ready model packaging.

## Final Model Performance

The selected model was trained using balanced sampling and full fine-tuning.

| Evaluation Metric | Test Score |
|---|---:|
| Accuracy | 43.70% |
| Macro Precision | 39.52% |
| Macro Recall | 41.56% |
| Macro F1-Score | 39.85% |

These results represent the model's performance on the test dataset used during project evaluation. The test set was inspected during multiple experimentation stages, so the results should not be considered a fully independent benchmark.

## Key Project Achievements

- Inspected and prepared a dataset containing 23 skin disease categories.
- Implemented image preprocessing and data augmentation.
- Developed an EfficientNet-B0 transfer learning pipeline.
- Applied selective and full fine-tuning strategies.
- Addressed class imbalance using weighted random sampling.
- Compared multiple model checkpoints using validation metrics.
- Evaluated classification performance using accuracy, precision, recall, macro F1-score, and classification reports.
- Exported the selected model into a reusable inference package.
- Verified the exported inference pipeline using a real test image.

## Exported Model Package

The final package includes:

- `model.pth` — Trained model weights.
- `metadata.json` — Model configuration, class names, preprocessing details, and evaluation metrics.
- `inference.py` — Reusable prediction interface.
- `requirements.txt` — Required Python dependencies.

A ZIP archive is also generated for convenient distribution.

## Limitations and Future Improvements

The current model has limited classification performance, indicating opportunities for improvement.

Potential future work includes:

- Increasing dataset diversity and image quality.
- Investigating class-wise performance and difficult disease categories.
- Experimenting with alternative architectures.
- Improving hyperparameter tuning and regularization.
- Evaluating on a completely independent external dataset.
- Developing a user-friendly application for model inference.

## Important Disclaimer

This project is intended strictly for research and educational purposes. It is not a medical diagnostic system and must not be used to diagnose diseases, recommend treatments, or make clinical decisions. Predictions require professional medical interpretation.

---

**Project Status: Completed — Model Training, Evaluation, Export, and Inference Verification.**